# Flujo final: predicción de cancelaciones hoteleras

Este notebook resume el sistema definitivo. No reentrena modelos ni vuelve a evaluar el test de 2017, que permanece cerrado.

## Decisiones principales

- Predicción al confirmar una reserva.
- Separación temporal: entrenamiento hasta septiembre de 2016, validación en el último trimestre y test en 2017.
- Se excluyen fugas directas y datos posteriores a la reserva.
- XGBoost se seleccionó por F1 de validación con umbral 0,3619.

In [ ]:
from pathlib import Path
import pandas as pd

from src.hotel_cancellation import load_dataset, remove_exact_duplicates, split_by_arrival_date

DATA_PATH = Path('data/raw/dataset_practica_final.csv')
data = load_dataset(DATA_PATH)
data, removed = remove_exact_duplicates(data)
splits = split_by_arrival_date(data)

print(f'Registros después de deduplicar: {len(data):,}')
print(f'Duplicados exactos eliminados: {removed:,}')
for name, subset in [('train', splits.train), ('validation', splits.validation), ('test_2017_cerrado', splits.test)]:
    print(f'{name}: {len(subset):,} filas')

## Resultado seleccionado

XGBoost obtuvo F1 de validación **0,6218** y se evaluó una única vez en 2017: F1 **0,6280**, ROC-AUC **0,8098** y recall **0,8289**.

La celda siguiente sirve solo para aplicar el modelo local a nuevas reservas; no utiliza `is_canceled` ni calcula métricas.

In [ ]:
from src.hotel_cancellation.inference import load_final_model, predict_cancellations

MODEL_PATH = Path('artifacts/model_optimization/models/xgboost.joblib')
model = load_final_model(MODEL_PATH)

# Sustituir por un CSV de nuevas reservas con las columnas disponibles al confirmar.
new_reservations = data.head(3).drop(columns=['is_canceled'])
predictions = predict_cancellations(model, new_reservations)
predictions

## Cierre

Antes de entregar, ejecutar `python -m scripts.run_pipeline check -- --require-data --require-model` y revisar la documentación de `docs/CIERRE_TECNICO.md`. Los datos, modelos y resultados locales se mantienen fuera de Git.